In [ ]:
"""
Scones Unlimited - Lambda functions for the image classification workflow.
Each function was deployed as its own AWS Lambda (Python 3.12) with the handler
name `lambda_handler`; they are renamed here only so all three fit in one file.

Workflow: serializeImageData -> classifyImage -> filterLowConfidence
"""
import json
import base64
import boto3

s3 = boto3.client("s3")
runtime = boto3.client("sagemaker-runtime")

# ---------------------------------------------------------------------------
# Lambda 1: serializeImageData
# Downloads an image from S3 and returns it base64-encoded as `image_data`.
# ---------------------------------------------------------------------------
def serialize_image_data_handler(event, context):
    key = event["s3_key"]                               # e.g. "test/bicycle_s_000513.png"
    bucket = event["s3_bucket"]

    s3.download_file(bucket, key, "/tmp/image.png")     # /tmp is Lambda's only writable folder

    with open("/tmp/image.png", "rb") as f:
        image_data = base64.b64encode(f.read())         # PNG bytes -> base64 bytes

    print("Event:", event.keys())
    return {
        "statusCode": 200,
        "body": {
            "image_data": image_data.decode("utf-8"),   # base64 bytes -> text (JSON-serializable)
            "s3_bucket": bucket,
            "s3_key": key,
            "inferences": [],
        },
    }


# ---------------------------------------------------------------------------
# Lambda 2: classifyImage
# Decodes the image and gets class probabilities from the SageMaker endpoint.
# Uses boto3 (built into Lambda) instead of packaging the SageMaker SDK.
# ---------------------------------------------------------------------------
ENDPOINT = "image-classification-2026-11-20-11-24-15-559"

def classify_image_handler(event, context):
    image = base64.b64decode(event["image_data"])       # base64 text -> PNG bytes

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT,
        ContentType="image/png",
        Body=image,
    )
    inferences = response["Body"].read().decode("utf-8")   # b'[0.96, 0.04]' -> '[0.96, 0.04]'

    event["inferences"] = json.loads(inferences)        # -> [0.96, 0.04]  (index 0 = bicycle, 1 = motorcycle)
    return {"statusCode": 200, "body": event}


# ---------------------------------------------------------------------------
# Lambda 3: filterLowConfidence
# Passes the event on only if the top probability meets the threshold;
# otherwise raises an error so the Step Function fails loudly.
# ---------------------------------------------------------------------------
THRESHOLD = 0.93

def filter_low_confidence_handler(event, context):
    inferences = event["inferences"]
    meets_threshold = max(inferences) >= THRESHOLD

    if not meets_threshold:
        raise Exception("THRESHOLD_CONFIDENCE_NOT_MET")

    return {"statusCode": 200, "body": event}